EDA

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/listings.csv.gz')
len(df)

4961

In [21]:
df.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'name', 'description',
       'picture_url', 'host_id', 'host_url', 'host_profile_id',
       'host_profile_url', 'host_name', 'hosts_time_as_user_years',
       'hosts_time_as_user_months', 'hosts_time_as_host_years',
       'hosts_time_as_host_months', 'host_location', 'host_about',
       'host_is_superhost', 'host_picture_url', 'host_listings_count',
       'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type',
       'room_type', 'accommodates', 'bathrooms_text', 'bedrooms', 'beds',
       'amenities', 'price', 'price_quote_checkin_date',
       'price_quote_checkout_date', 'price_quote_total_price',
       'price_quote_price_per_night', 'price_quote_raw', 'minimum_nights',
       'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights',
       'minimum_maximum_nights', 'maximum_maximum_nights',
       'minimum_nights_avg_ntm', 'maximum_nights_

In [3]:
df.head().T

,0,1,2,3,4
id,1434108413206019816,1613802572207908117,1674132228660847237,43057420,43057598
listing_url,https://www.airbnb.com/rooms/1434108413206019816,https://www.airbnb.com/rooms/1613802572207908117,https://www.airbnb.com/rooms/1674132228660847237,https://www.airbnb.com/rooms/43057420,https://www.airbnb.com/rooms/43057598
scrape_id,20260922161643,20260922161643,20260922161643,20260922161643,20260922161643
last_scraped,2026-09-24,2026-09-24,2026-09-24,2026-09-24,2026-09-24
source,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
calculated_host_listings_count,1,212,1,13,13
calculated_host_listings_count_entire_homes,1,211,1,13,13
calculated_host_listings_count_private_rooms,0,1,0,0,0
calculated_host_listings_count_shared_rooms,0,0,0,0,0


Check the missing values

In [4]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

neighborhood_overview           4961
host_since                      4961
host_acceptance_rate            4961
host_response_rate              4961
host_response_time              4961
host_total_listings_count       4961
host_neighbourhood              4961
host_verifications              4961
host_thumbnail_url              4961
calendar_updated                4961
bathrooms                       4961
neighbourhood_group_cleansed    4961
neighbourhood                   4961
instant_bookable                4961
source                          4526
host_about                      2750
host_location                   1855
first_review                     571
reviews_per_month                571
review_scores_checkin            571
review_scores_accuracy           571
review_scores_rating             571
review_scores_cleanliness        571
review_scores_communication      571
review_scores_location           571
review_scores_value              571
last_review                      571
e

Delete data with empty or almost empty columns  

In [5]:
columns_empty = missing[missing > len(df)*0.9].index
df = df.drop(columns=columns_empty)
len(df)

4961

In [6]:
columns_empty

Index(['source', 'neighborhood_overview', 'host_since', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url',
       'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
       'neighbourhood', 'neighbourhood_group_cleansed', 'bathrooms',
       'calendar_updated', 'instant_bookable'],
      dtype='str')

Delete data with missing price

In [7]:
df = df.drop(df[df.price.isnull()].index)
len(df)

4627

In [8]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

host_about                     2544
host_location                  1717
review_scores_communication     501
review_scores_location          501
first_review                    501
review_scores_rating            501
last_review                     501
reviews_per_month               501
review_scores_value             501
review_scores_cleanliness       501
review_scores_accuracy          501
review_scores_checkin           501
license                          93
beds                             47
has_availability                  6
minimum_minimum_nights            3
maximum_minimum_nights            3
minimum_nights                    3
maximum_nights                    3
minimum_maximum_nights            3
maximum_maximum_nights            3
bathrooms_text                    1
description                       1
dtype: int64

Delete some unfinished offers. First to check if it the same raws

In [9]:
cols = missing[missing == 3].index
df[cols].isnull().any(axis=1).sum()

np.int64(3)

In [10]:
df = df.dropna(subset=cols)
len(df)

4624

Dropping the raw with bathrom description missing

In [11]:
df = df.drop(df[df.bathrooms_text.isnull()].index)
len(df)

4623

Analyse whats left

In [12]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

host_about                     2541
host_location                  1716
last_review                     501
first_review                    501
review_scores_rating            501
review_scores_checkin           501
review_scores_cleanliness       501
review_scores_accuracy          501
review_scores_location          501
review_scores_value             501
reviews_per_month               501
review_scores_communication     501
license                          92
beds                             47
has_availability                  6
description                       1
dtype: int64

In [ ]:
df['host_location'].value_counts()

Feature engineering

1. Host location (categories)

In [19]:
def host_location_category(loc): 
    if pd.isna(loc): 
        return 'unknown' 
    if 'Thessaloniki' in loc: 
        return 'thessaloniki' 
    if'Greece' in loc: 
        return 'greece'
    return 'abroad' 
    
df['host_location_cat'] = df['host_location'].apply(host_location_category) 

In [20]:
df['host_location_cat'].value_counts()

host_location_cat
thessaloniki    2410
unknown         1716
greece           372
abroad           125
Name: count, dtype: int64

2. Reviews
   a) make a new column 'has_reviews'
   b) 'reviews_per_month    ' fill with 0

In [28]:
def has_reviews_category(loc): 
    if loc == 0: 
        return 0 
    return 1 
    
df['has_reviews'] = df['number_of_reviews'].apply(has_reviews_category) 

In [29]:
df['has_reviews'].value_counts()

has_reviews
1    4122
0     501
Name: count, dtype: int64

In [30]:
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

In [31]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

host_about                     2541
host_location                  1716
last_review                     501
first_review                    501
review_scores_cleanliness       501
review_scores_accuracy          501
review_scores_rating            501
review_scores_communication     501
review_scores_location          501
review_scores_value             501
review_scores_checkin           501
license                          92
beds                             47
has_availability                  6
description                       1
dtype: int64

3. License

In [32]:
df['license'].value_counts()

license
Exempt              36
0933Κ13000111100    15
0933Κ13000118900    15
0933Κ13000125400    11
1037314             10
                    ..
00003955113          1
36234670000          1
0933Κ13000130600     1
00003652492          1
00003941335          1
Name: count, Length: 4244, dtype: int64

In [36]:
def license_ok(loc): 
    if pd.isna(loc): 
        return 0
    return 1 
    
df['license_ok'] = df['license'].apply(license_ok) 

In [37]:
df['license_ok'].value_counts()

license_ok
1    4531
0      92
Name: count, dtype: int64

In [41]:
df['has_availability'].value_counts()

has_availability
t    4617
Name: count, dtype: int64

Delete non unique columns

In [44]:
df.nunique().sort_values(ascending=True)

scrape_id                    1
has_availability             1
host_has_profile_pic         2
last_scraped                 2
host_identity_verified       2
                          ... 
license                   4244
name                      4552
picture_url               4574
id                        4623
listing_url               4623
Length: 78, dtype: int64

In [53]:
nunique = df.nunique()
columns_nonunique = nunique[nunique == 1].index
df = df.drop(columns=columns_nonunique)
